In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats
import os
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

In [9]:
sample_key = pd.read_excel('../../screening_data/sample_key.xlsx')

#exclude CDK12/13 for now
count_holder = []

for i, val in sample_key.iterrows():
    sample = val['Sample']
    file_name = val['File_Name']
    editor = val['Editor']
    data_loc = val['Screen_data_location']
    subpool = val['Subpool']

    #get the directory and the relevant files
    j = os.listdir(f'../../screening_data/{data_loc}_screen_data/counts')
    file_list = [i for i in j if file_name in i]

    #extract and combine files (data split by sequencer for same sample)
    a = pd.read_csv(f'../../screening_data/{data_loc}_screen_data/counts/{file_list[0]}')
    b = pd.read_csv(f'../../screening_data/{data_loc}_screen_data/counts/{file_list[1]}')

    a['total_guide_count'] = a['total_guide_count'] + b['total_guide_count']
    a['matched_guide_count'] = a['matched_guide_count'] + b['matched_guide_count']
    a['bc_count'] = a['bc_count'] + b['bc_count']

    count_holder.append(a)

count_dict = dict(zip(sample_key['File_Name'], count_holder))

In [49]:
#double check that ordering is the same for all tables
for i in count_dict.keys():
    assert list(count_dict['D24-257001']['Guide_ID'])==list(count_dict[i]['Guide_ID'])


# Generate count tables for each screen

- Include plasmid library in each table
- Include CBE replicate included in ABE data
- Generate barcode counts, proto counts, and matched guide counts tables

# 1. CBE Subpool 1

In [20]:
CBE = sample_key[sample_key['Screen_data_location']=='CBE_subpool_1']
missing_replicate = sample_key[(sample_key['Screen_data_location']=='ABE_subpool_1') & (sample_key['Editor']=='CBE')]

CBE1 = pd.concat((CBE, missing_replicate))
CBE1

,File_Name,Sample,Editor,Screen_data_location,Subpool
0,D24-257001,Plasmid,PLASMID,CBE_subpool_1,1
1,D24-257002,T0_REP1,CBE,CBE_subpool_1,1
2,D24-257003,T0_REP2,CBE,CBE_subpool_1,1
3,D24-257004,DMSO_REP1,CBE,CBE_subpool_1,1
4,D24-257005,DMSO_REP2,CBE,CBE_subpool_1,1
5,D24-257006,DMSO_REP3,CBE,CBE_subpool_1,1
6,D24-257007,KI-CDK9d-32_100nM_REP1,CBE,CBE_subpool_1,1
7,D24-257008,KI-CDK9d-32_100nM_REP2,CBE,CBE_subpool_1,1
8,D24-257009,KI-CDK9d-32_100nM_REP3,CBE,CBE_subpool_1,1
9,D24-257010,KI-CDK9d-32_1000nM_REP1,CBE,CBE_subpool_1,1


In [ ]:
#function for generating the count tables
#takes in the subpool information as well to select relevant gRNAs

def count_generator(sample_table, count_dict, library, pool='F1-R1'):
    """ 
    sample_table = table of samples to put into counts table (see above for example)
    count_dict = dictionary of count tables (generated at start of notebook)
    library = library file (CDK_library_final.csv)
    pool = 'F1-R1', "F2-R2", 'F3-R3'
    """

    subpool = library[library['Pool']==pool]
    subpool_guides = list(subpool['gRNA_id'])

    guide_holder = []
    bc_count_holder = []
    proto_count_holder = []
    matched_count_holder = []
    name_holder = []
    
    for i, val in sample_table.iterrows():

        file = val['File_Name']
        name = val['Sample']

        count1 = count_dict[file]

        count_subpool = count1[count1['Guide_ID'].isin(subpool_guides)]

        bc_count_holder.append(list(count_subpool['bc_count']))
        proto_count_holder.append(list(count_subpool['total_guide_count']))
        matched_count_holder.append(list(count_subpool['matched_guide_count']))
        guide_holder.append(list(count_subpool['Guide_ID']))
        name_holder.append(name)

    #double check that the ordering is correct
    for i in guide_holder:
        assert guide_holder[0]==i

    #make the tables
    col_names = ['gRNA_id'] + name_holder

    bc_count_df = pd.DataFrame(dict(zip(col_names, [guide_holder[0]] + bc_count_holder)))
    proto_count_df = pd.DataFrame(dict(zip(col_names, [guide_holder[0]] + proto_count_holder)))
    matched_count_df = pd.DataFrame(dict(zip(col_names, [guide_holder[0]] + matched_count_holder)))

    return bc_count_df, proto_count_df, matched_count_df


In [76]:
library = pd.read_csv('../../source_data/02_library/CDK_library_final.csv')

bc_count_df, proto_count_df, matched_count_df = count_generator(CBE1, count_dict, library, pool='F1-R1')

#and then save it
name = 'CBE_subpool1'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)


# 2. ABE Subpool 1

In [78]:
ABE = sample_key[(sample_key['Screen_data_location']=='ABE_subpool_1') & (sample_key['Editor']=='ABE')]
missing_plasmid = sample_key[(sample_key['Screen_data_location']=='CBE_subpool_1') & (sample_key['Editor']=='PLASMID')]

ABE1 = pd.concat((ABE, missing_plasmid))
ABE1

,File_Name,Sample,Editor,Screen_data_location,Subpool
30,D24-295001,T0_REP1,ABE,ABE_subpool_1,1
31,D24-295002,T0_REP2,ABE,ABE_subpool_1,1
32,D24-295003,T0_REP3,ABE,ABE_subpool_1,1
33,D24-295004,DMSO_REP1,ABE,ABE_subpool_1,1
34,D24-295005,DMSO_REP2,ABE,ABE_subpool_1,1
35,D24-295006,DMSO_REP3,ABE,ABE_subpool_1,1
36,D24-295007,KI-CDK9d-32_100nM_REP1,ABE,ABE_subpool_1,1
37,D24-295008,KI-CDK9d-32_100nM_REP2,ABE,ABE_subpool_1,1
38,D24-295009,KI-CDK9d-32_100nM_REP3,ABE,ABE_subpool_1,1
39,D24-295010,KI-CDK9d-32_1000nM_REP1,ABE,ABE_subpool_1,1


In [79]:
bc_count_df, proto_count_df, matched_count_df = count_generator(ABE1, count_dict, library, pool='F1-R1')

#and then save it
name = 'ABE_subpool1'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)

# 3. CBE SY-5609

In [80]:
missing_plasmid = sample_key[(sample_key['Screen_data_location']=='CBE_subpool_1') & (sample_key['Editor']=='PLASMID')]
SY_CBE = sample_key[(sample_key['Screen_data_location']=='SY_5609') & (sample_key['Editor']=='CBE')]

SY_CBE1 = pd.concat((SY_CBE, missing_plasmid))
SY_CBE1

,File_Name,Sample,Editor,Screen_data_location,Subpool
61,D25-148001,CBE_T0_REP1,CBE,SY_5609,1
62,D25-148002,CBE_T0_REP2,CBE,SY_5609,1
63,D25-148003,CBE_T0_REP3,CBE,SY_5609,1
64,D25-148004,CBE_DMSO_REP1,CBE,SY_5609,1
65,D25-148005,CBE_DMSO_REP2,CBE,SY_5609,1
66,D25-148006,CBE_DMSO_REP3,CBE,SY_5609,1
67,D25-148007,CBE_SY-5609_10nM_REP1,CBE,SY_5609,1
68,D25-148008,CBE_SY-5609_10nM_REP2,CBE,SY_5609,1
69,D25-148009,CBE_SY-5609_10nM_REP3,CBE,SY_5609,1
70,D25-148010,CBE_SY-5609_100nM_REP1,CBE,SY_5609,1


In [81]:
bc_count_df, proto_count_df, matched_count_df = count_generator(SY_CBE1, count_dict, library, pool='F1-R1')

#and then save it
name = 'SY_5609_CBE'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)

# 4. ABE SY-5609

In [82]:
missing_plasmid = sample_key[(sample_key['Screen_data_location']=='CBE_subpool_1') & (sample_key['Editor']=='PLASMID')]
SY_ABE = sample_key[(sample_key['Screen_data_location']=='SY_5609') & (sample_key['Editor']=='ABE')]

SY_ABE1 = pd.concat((SY_CBE, missing_plasmid))
SY_ABE1


,File_Name,Sample,Editor,Screen_data_location,Subpool
61,D25-148001,CBE_T0_REP1,CBE,SY_5609,1
62,D25-148002,CBE_T0_REP2,CBE,SY_5609,1
63,D25-148003,CBE_T0_REP3,CBE,SY_5609,1
64,D25-148004,CBE_DMSO_REP1,CBE,SY_5609,1
65,D25-148005,CBE_DMSO_REP2,CBE,SY_5609,1
66,D25-148006,CBE_DMSO_REP3,CBE,SY_5609,1
67,D25-148007,CBE_SY-5609_10nM_REP1,CBE,SY_5609,1
68,D25-148008,CBE_SY-5609_10nM_REP2,CBE,SY_5609,1
69,D25-148009,CBE_SY-5609_10nM_REP3,CBE,SY_5609,1
70,D25-148010,CBE_SY-5609_100nM_REP1,CBE,SY_5609,1


In [83]:
bc_count_df, proto_count_df, matched_count_df = count_generator(SY_ABE1, count_dict, library, pool='F1-R1')

#and then save it
name = 'SY_5609_ABE'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)

# 5. CBE Compound Mutant (KB-0742) Screen

In [84]:
missing_plasmid = sample_key[(sample_key['Screen_data_location']=='CBE_subpool_1') & (sample_key['Editor']=='PLASMID')]
compound = sample_key[(sample_key['Screen_data_location']=='KB_compound_mut')]
compound1 = pd.concat((compound, missing_plasmid))
compound1

,File_Name,Sample,Editor,Screen_data_location,Subpool
85,D25-173001,T0_REP1,CBE,KB_compound_mut,1
86,D25-173002,T0_REP2,CBE,KB_compound_mut,1
87,D25-173003,T0_REP3,CBE,KB_compound_mut,1
88,D25-173004,DMSO_REP1,CBE,KB_compound_mut,1
89,D25-173005,DMSO_REP2,CBE,KB_compound_mut,1
90,D25-173006,DMSO_REP3,CBE,KB_compound_mut,1
91,D25-173007,KB_2000_REP1,CBE,KB_compound_mut,1
92,D25-173008,KB_2000_REP2,CBE,KB_compound_mut,1
93,D25-173009,KB_2000_REP3,CBE,KB_compound_mut,1
94,D25-173010,KB_4000_REP1,CBE,KB_compound_mut,1


In [85]:
bc_count_df, proto_count_df, matched_count_df = count_generator(compound1, count_dict, library, pool='F1-R1')

#and then save it
name = 'Compound_mutant'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)

# 6. CBE CDK12/13 Subpool 2

In [88]:
CDK12_13_CBE = sample_key[(sample_key['Screen_data_location']=='CDK12_13') & (sample_key['Editor'].isin(['CBE', 'PLASMID']))]
CDK12_13_CBE

,File_Name,Sample,Editor,Screen_data_location,Subpool
97,D25-188001,CBE_T0_REP1,CBE,CDK12_13,2
98,D25-188002,CBE_T0_REP2,CBE,CDK12_13,2
99,D25-188003,CBE_T0_REP3,CBE,CDK12_13,2
100,D25-188004,CBE_DMSO_REP1,CBE,CDK12_13,2
101,D25-188005,CBE_DMSO_REP2,CBE,CDK12_13,2
102,D25-188006,CBE_DMSO_REP3,CBE,CDK12_13,2
103,D25-188007,CBE_BSJ-4-116_REP1,CBE,CDK12_13,2
104,D25-188008,CBE_BSJ-4-116_REP2,CBE,CDK12_13,2
105,D25-188009,CBE_BSJ-4-116_REP3,CBE,CDK12_13,2
106,D25-188010,CBE_CDK12-IN-2_REP1,CBE,CDK12_13,2


In [89]:
bc_count_df, proto_count_df, matched_count_df = count_generator(CDK12_13_CBE, count_dict, library, pool='F2-R2')

#and then save it
name = 'CDK12_13_CBE'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)

# 7. ABE CDK12/13 Subpool 2

In [91]:
CDK12_13_ABE = sample_key[(sample_key['Screen_data_location']=='CDK12_13') & (sample_key['Editor'].isin(['ABE', 'PLASMID']))]
CDK12_13_ABE

,File_Name,Sample,Editor,Screen_data_location,Subpool
112,D25-188016,ABE_T0_REP1,ABE,CDK12_13,2
113,D25-188017,ABE_T0_REP2,ABE,CDK12_13,2
114,D25-188018,ABE_T0_REP3,ABE,CDK12_13,2
115,D25-188019,ABE_DMSO_REP1,ABE,CDK12_13,2
116,D25-188020,ABE_DMSO_REP2,ABE,CDK12_13,2
117,D25-188021,ABE_DMSO_REP3,ABE,CDK12_13,2
118,D25-188022,ABE_BSJ-4-116_REP1,ABE,CDK12_13,2
119,D25-188023,ABE_BSJ-4-116_REP2,ABE,CDK12_13,2
120,D25-188024,ABE_BSJ-4-116_REP3,ABE,CDK12_13,2
121,D25-188025,ABE_CDK12-IN-2_REP1,ABE,CDK12_13,2


In [92]:
bc_count_df, proto_count_df, matched_count_df = count_generator(CDK12_13_ABE, count_dict, library, pool='F2-R2')

#and then save it
name = 'CDK12_13_ABE'
bc_count_df.to_csv(f'../../screening_data/01_count_tables/barcode_counts/{name}_bc_counts.csv', index=False)
proto_count_df.to_csv(f'../../screening_data/01_count_tables/proto_counts/{name}_proto_counts.csv', index=False)
matched_count_df.to_csv(f'../../screening_data/01_count_tables/matched_counts/{name}_matched_counts.csv', index=False)